# reload.ipynb

Refresh **this** `Pod_N` bundle from the repo's `Pod/` template. Bundles are
snapshots made by `prepare_pods.ipynb`; when the template changes on GitHub they
go stale. Running this notebook:

1. FORCE-syncs the repo to `origin/main` (clones if missing);
2. copies the template's files over this bundle (`executed_*` outputs and other
   local extras are kept; template files are overwritten -- including this
   notebook itself, which is safe: the running kernel is unaffected);
3. re-stamps this bundle's `VM_NAME` (read from the current `training.ipynb`,
   else derived from the folder name `Pod_N` -> `VMN`).

Safe to re-run any time. Afterwards close + reopen any bundle notebooks you had
open so Jupyter reloads the refreshed copies from disk.

In [ ]:
# Pull origin/main, then re-materialise this bundle from the Pod/ template.
import json, os, re, shutil, subprocess
from pathlib import Path

URL = 'https://github.com/Nice9Tian/stable-query-latent.git'
REPO = '/workspace/stable-query-latent'

BUNDLE = Path.cwd()
assert re.fullmatch(r'Pod_\d+', BUNDLE.name), (
    f'run this from a generated bundle (/workspace/Pod_N), not {BUNDLE}')

def sh(cmd):
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=False)

sh('type -p git >/dev/null 2>&1 || (apt-get update && apt-get install -y git)')
if not os.path.isdir(os.path.join(REPO, '.git')):
    sh(f'git clone {URL} {REPO}')
sh(f'cd {REPO} && git remote set-url origin {URL} && git fetch origin main && git reset --hard origin/main')
sh(f'cd {REPO} && git rev-parse --short HEAD')

TEMPLATE = Path(REPO) / 'Pod'
assert TEMPLATE.is_dir(), f'template not found: {TEMPLATE} (did the pull succeed?)'
# Keep in sync with prepare_pods.ipynb's NBS_WITH_VMNAME.
NBS_WITH_VMNAME = ['training.ipynb', 'prepare_training.ipynb', 'realtime_reader.ipynb', 'eval_curve.ipynb']
VM_RE = re.compile(r'VM_NAME = "([^"]*)"')

# Preserve this bundle's identity: read VM_NAME from the current training.ipynb
# (parse the JSON -- a raw-text regex would miss the escaped quotes), falling
# back to the folder-name convention Pod_N -> VMN.
def read_vm_name(nb_path):
    try:
        doc = json.loads(Path(nb_path).read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return None
    for cell in doc.get('cells', []):
        src = cell.get('source', '')
        joined = ''.join(src) if isinstance(src, list) else src
        m = VM_RE.search(joined)
        if m:
            return m.group(1)
    return None

vm = read_vm_name(BUNDLE / 'training.ipynb')
if not vm or vm == 'vmA':          # missing or template placeholder -> derive
    vm = 'VM' + BUNDLE.name.split('_', 1)[1]

shutil.copytree(TEMPLATE, BUNDLE, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('__pycache__', 'Pod_[0-9]*'))

def set_vm_name(nb_path, vm):
    doc = json.loads(Path(nb_path).read_text(encoding='utf-8'))
    hits = 0
    for cell in doc.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        src = cell['source']
        joined = ''.join(src) if isinstance(src, list) else src
        if VM_RE.search(joined):
            cell['source'] = VM_RE.sub(f'VM_NAME = "{vm}"', joined, count=1)
            hits += 1
    Path(nb_path).write_text(json.dumps(doc, ensure_ascii=False, indent=1) + '\n', encoding='utf-8')
    return hits

for nb in NBS_WITH_VMNAME:
    h = set_vm_name(BUNDLE / nb, vm)
    assert h == 1, f'{BUNDLE / nb}: expected 1 VM_NAME line, found {h}'

print(f'\nreloaded {BUNDLE} from {TEMPLATE}  (VM_NAME={vm})')
print('close + reopen any bundle notebooks you had open (Jupyter still shows the old buffer).')